# Milestone 2 - Q8: baseline 3-stage U-Net, 5 epochs

Trains a 3-stage U-Net (3 input channels, 4 output channels) on the Paradigm 1 Dataset for 5 epochs with AdamW (lr 3e-4), logs train and validation loss every epoch to Weights & Biases, and keeps the checkpoint with the lowest validation loss. Same stratified holdout split as the rest of the project.

## Setup

In [1]:
import os, glob, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

IMG_H, IMG_W = 128, 800
DEFECTS = ["defect_a", "defect_b", "defect_c", "defect_d"]

KNOWN_TRAIN = "/kaggle/input/competitions/industrial-surface-defect-detection/public/train.csv"

def find_train_csv():
    """Use the known Kaggle path; otherwise search /kaggle/input and the working folder."""
    if os.path.exists(KNOWN_TRAIN):
        return KNOWN_TRAIN
    for root in ["/kaggle/input", "."]:
        hits = glob.glob(os.path.join(root, "**", "train.csv"), recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError("train.csv not found - attach the competition data to this notebook")

TRAIN_CSV = find_train_csv()
print("Using:", TRAIN_CSV)

Using: /kaggle/input/competitions/industrial-surface-defect-detection/public/train.csv


In [2]:
import random, time
import torch
import torch.nn as nn

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "(no GPU: enable the accelerator in the notebook settings)")

device: cuda Tesla T4


## RLE decoder and Dataset (Paradigm 1, from Q7)

In [3]:
def rle_encode(mask):
    """Binary 2D mask -> RLE string. 1-based start index, column-major (Fortran) order."""
    flat = np.asarray(mask).flatten(order="F")          
    padded = np.concatenate([[0], flat, [0]])           
    k = np.where(padded[1:] != padded[:-1])[0] + 1      
    starts, ends = k[0::2], k[1::2]                     
    return " ".join(f"{s} {e - s}" for s, e in zip(starts, ends))

def rle_decode(rle, shape=(IMG_H, IMG_W)):
    """RLE string -> binary (H, W) mask. NaN / empty / blank string -> all zeros."""
    flat = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    if isinstance(rle, str) and rle.strip():
        tok = np.array(rle.split(), dtype=np.int64)
        starts = tok[0::2] - 1                          
        ends = starts + tok[1::2]
        for s, e in zip(starts, ends):
            flat[max(s, 0):min(e, flat.size)] = 1       
    return flat.reshape(shape, order="F")               

In [4]:
import cv2
import torch
from torch.utils.data import Dataset, DataLoader

def find_image_dir(sample_id):
    """Locate the folder that holds the training images (by looking for one known image)."""
    hits = glob.glob(f"/kaggle/input/**/{sample_id}.*", recursive=True) or glob.glob(f"./**/{sample_id}.*", recursive=True)
    if not hits:
        raise FileNotFoundError(f"image {sample_id} not found - attach the competition data")
    return os.path.dirname(hits[0])

class SurfaceAnomalyDataset(Dataset):
    """Paradigm 1: one sample = one image -> (3,128,800) float32 in [0,1], (4,128,800) float32 masks."""
    def __init__(self, df, img_dir):
        self.img_dir = img_dir
        
        wide = df.pivot(index="image_id", columns="defect_type", values="mask_rle").reindex(columns=DEFECTS)
        self.ids = wide.index.to_list()
        self.rles = wide.to_numpy(dtype=object)               
        self.files = {os.path.splitext(f)[0]: f for f in os.listdir(img_dir)}   

    def __len__(self):
        return len(self.ids)                                  

    def __getitem__(self, idx):
        path = os.path.join(self.img_dir, self.files[self.ids[idx]])
        bgr = cv2.imread(path, cv2.IMREAD_COLOR)              
        if bgr is None:
            raise FileNotFoundError(path)
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)            
        image = torch.from_numpy(rgb.astype(np.float32) / 255.0).permute(2, 0, 1).contiguous()  
        masks = np.stack([rle_decode(r) for r in self.rles[idx]]).astype(np.float32)            
        return image, torch.from_numpy(masks)

## Train / validation split
The stratified split saved in Milestone 1 (`holdout.csv`: `image_id`, `combo`, `split`) is reused so that all models are compared on the same validation images.

In [5]:
KNOWN_HOLDOUT = "/kaggle/input/notebooks/mruniksurya/milestone-1/holdout.csv"
holdout_path = KNOWN_HOLDOUT if os.path.exists(KNOWN_HOLDOUT) else glob.glob("/kaggle/input/**/holdout.csv", recursive=True)[0]
holdout = pd.read_csv(holdout_path)
print(holdout["split"].value_counts().to_dict())

is_val = holdout["split"].str.lower().isin({"val", "valid", "validation", "holdout"})
train_ids, val_ids = set(holdout.loc[~is_val, "image_id"]), set(holdout.loc[is_val, "image_id"])
assert train_ids.isdisjoint(val_ids) and len(val_ids) > 0
print("train images:", len(train_ids), "| validation images:", len(val_ids))

df = pd.read_csv(TRAIN_CSV)
IMG_DIR = find_image_dir(df["image_id"].iloc[0])
train_ds = SurfaceAnomalyDataset(df[df["image_id"].isin(train_ids)], IMG_DIR)
val_ds = SurfaceAnomalyDataset(df[df["image_id"].isin(val_ids)], IMG_DIR)

BATCH_SIZE = 16
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
x, y = next(iter(train_loader))
print("batch:", tuple(x.shape), x.dtype, tuple(y.shape), y.dtype)

{'train': 9114, 'val': 2279}
train images: 9114 | validation images: 2279
batch: (16, 3, 128, 800) torch.float32 (16, 4, 128, 800) torch.float32


## Model, loss and metric
- **3-stage U-Net**: three down stages (32, 64, 128 channels) with max-pooling, a 256-channel bottleneck, and three up stages that concatenate the matching encoder features (skip connections). The 1x1 head gives 4 independent logits per pixel.
- **Loss**: BCE-with-logits plus soft Dice on the (image, class) pairs that contain a defect. BCE alone is dominated by the 99% background pixels; the Dice term rewards overlap.
- **Metric**: official-style score at threshold 0.5, logged for information (the question itself asks for the loss).

In [6]:
import torch.nn as nn

class DoubleConv(nn.Module):
    """(Conv3x3 -> BatchNorm -> ReLU) x 2: the basic U-Net building block."""
    def __init__(self, cin, cout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))
    def forward(self, x):
        return self.block(x)

class UNet3Stage(nn.Module):
    """U-Net with 3 down stages + bottleneck. Input (B,in_ch,128,800) -> logits (B,4,128,800)."""
    def __init__(self, in_ch=3, out_ch=4, base=32):
        super().__init__()
        self.enc1 = DoubleConv(in_ch, base)
        self.enc2 = DoubleConv(base, base * 2)
        self.enc3 = DoubleConv(base * 2, base * 4)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(base * 4, base * 8)
        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.dec3 = DoubleConv(base * 8, base * 4)        
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = DoubleConv(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = DoubleConv(base * 2, base)
        self.head = nn.Conv2d(base, out_ch, 1)            

    def forward(self, x):
        e1 = self.enc1(x)                                 
        e2 = self.enc2(self.pool(e1))                     
        e3 = self.enc3(self.pool(e2))                     
        b = self.bottleneck(self.pool(e3))               
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))   
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.head(d1)                              
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [7]:
def soft_dice_loss(logits, targets, eps=1.0):
    """1 - soft Dice, averaged over (image, class) pairs that really contain a defect."""
    probs = torch.sigmoid(logits)
    inter = (probs * targets).sum(dim=(2, 3))                       
    denom = probs.sum(dim=(2, 3)) + targets.sum(dim=(2, 3))
    dice = (2 * inter + eps) / (denom + eps)
    positive = targets.sum(dim=(2, 3)) > 0
    if positive.any():
        return (1 - dice[positive]).mean()
    return logits.sum() * 0.0                                       

class BCEDiceLoss(nn.Module):
    """BCE-with-logits (pixel-wise) + soft Dice (overlap). Same loss as the main M2 pipeline."""
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
    def forward(self, logits, targets):
        return self.bce(logits, targets) + soft_dice_loss(logits, targets)

In [8]:
CANVAS_PIXELS = IMG_H * IMG_W

def dice_empty_aware(inter, pred_area, gt_area):
    inter, pred_area, gt_area = (np.asarray(a, dtype=np.float64) for a in (inter, pred_area, gt_area))
    denom = pred_area + gt_area
    dice = np.where(denom > 0, 2 * inter / np.maximum(denom, 1), 0.0)
    return np.where((pred_area == 0) & (gt_area == 0), 1.0, dice)

def competition_score(inter, pred_area, gt_area, canvas=CANVAS_PIXELS, weights=(0.70, 0.20, 0.10)):
    inter, pred_area, gt_area = (np.asarray(a, dtype=np.float64) for a in (inter, pred_area, gt_area))
    positive = gt_area > 0
    empty = ~positive
    dice = dice_empty_aware(inter, pred_area, gt_area)
    fg = dice[positive].mean()
    fp = 1 - (pred_area[empty] / canvas).mean()
    ea = (pred_area[empty] == 0).mean()
    return {"foreground_dice": fg, "false_positive_score": fp, "empty_accuracy": ea,
            "score": weights[0] * fg + weights[1] * fp + weights[2] * ea}

In [9]:
model = UNet3Stage(in_ch=3, out_ch=4, base=32).to(DEVICE)
print("parameters:", f"{count_params(model):,}")
with torch.no_grad():
    out = model(x[:2].to(DEVICE))
print("forward check:", tuple(out.shape))
assert out.shape == (2, 4, IMG_H, IMG_W)

parameters: 1,927,108
forward check: (2, 4, 128, 800)


## Training and validation loops

In [10]:
@torch.no_grad()
def evaluate(model, loader, loss_fn, device, tau=0.5):
    """Validation loss + official-style score at threshold tau (model in eval mode, no gradients)."""
    model.eval()
    total, n, I, P_, G = 0.0, 0, [], [], []
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        logits = model(x)
        total += loss_fn(logits, y).item() * x.size(0)
        n += x.size(0)
        pred, gt = torch.sigmoid(logits) > tau, y > 0.5
        I.append((pred & gt).sum(dim=(2, 3)).cpu().numpy().ravel())
        P_.append(pred.sum(dim=(2, 3)).cpu().numpy().ravel())
        G.append(gt.sum(dim=(2, 3)).cpu().numpy().ravel())
    m = competition_score(np.concatenate(I), np.concatenate(P_), np.concatenate(G))
    return total / n, m

def train_one_epoch(model, loader, loss_fn, optimizer, device):
    model.train()
    total, n = 0.0, 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)     
        loss = loss_fn(model(x), y)               
        loss.backward()                           
        optimizer.step()                          
        total += loss.item() * x.size(0)
        n += x.size(0)
    return total / n

## Weights & Biases run
The key is read from the Kaggle Secret `WANDB_API_KEY` (Add-ons > Secrets; internet must be on for this notebook). It is never written in the notebook.

In [12]:
import wandb
from kaggle_secrets import UserSecretsClient

wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))

CONFIG = dict(model="UNet3Stage", base=32, in_ch=3, out_ch=4, epochs=5, batch_size=BATCH_SIZE, lr=3e-4,
              weight_decay=1e-4, optimizer="AdamW", loss="BCEWithLogits+SoftDice", augmentation="none",
              input="RGB/255", seed=42, train_images=len(train_ds), val_images=len(val_ds))
run = wandb.init(entity="22f2001613-indian-institute-of-technology-madras", project="22f2001613-t32026",
                 name="m2-q8-unet-3stage-5ep", config=CONFIG, tags=["milestone-2", "q8", "scratch"])

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 22f2001613 (22f2001613-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: WARNING Fatal error while uploading data. Some run data will not be synced, but it will still be written to disk. Use `wandb sync` at the end of the run to try uploading.


## Train 5 epochs
After every epoch: training loss (mean over the epoch's batches), validation loss, validation score. The checkpoint with the **lowest validation loss** is saved as `best_unet_q8.pt`.

In [ ]:
loss_fn = BCEDiceLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])

history, best_val = [], float("inf")
for epoch in range(1, CONFIG["epochs"] + 1):
    t0 = time.time()
    tr_loss = train_one_epoch(model, train_loader, loss_fn, optimizer, DEVICE)
    va_loss, m = evaluate(model, val_loader, loss_fn, DEVICE, tau=0.5)
    history.append(dict(epoch=epoch, train_loss=tr_loss, val_loss=va_loss, val_score=m["score"], val_fg_dice=m["foreground_dice"]))
    wandb.log({"epoch": epoch, "train_loss": tr_loss, "val_loss": va_loss, "val_score_tau0.5": m["score"],
               "val_fg_dice": m["foreground_dice"], "val_empty_acc": m["empty_accuracy"]}, step=epoch)
    if va_loss < best_val:
        best_val = va_loss
        torch.save(model.state_dict(), "best_unet_q8.pt")
    print(f"epoch {epoch}: train {tr_loss:.4f} | val {va_loss:.4f} | score@0.5 {m['score']:.4f} | fgDice {m['foreground_dice']:.4f} | {time.time() - t0:.0f}s")

## Results and answers

In [ ]:
hist = pd.DataFrame(history).set_index("epoch")
print(hist.round(4).to_string())

best_epoch = int(hist["val_loss"].idxmin())
e1, e5 = hist.loc[1], hist.loc[CONFIG["epochs"]]
print(f"\nQ8 epoch1_(train, val): ({e1.train_loss:.4f}, {e1.val_loss:.4f})")
print(f"Q8 epoch5_(train, val): ({e5.train_loss:.4f}, {e5.val_loss:.4f})")
print(f"Q8 best_epoch: {best_epoch}")

gap = e5.val_loss - e5.train_loss
still_falling = hist["val_loss"].iloc[-1] < hist["val_loss"].iloc[-2] and hist["train_loss"].iloc[-1] < hist["train_loss"].iloc[-2]
print(f"final gap (val - train): {gap:+.4f} | both losses still falling at the last epoch: {still_falling}")

wandb.summary.update({"best_epoch": best_epoch, "best_val_loss": float(hist["val_loss"].min())})
fig, ax = plt.subplots(figsize=(5, 3.2))
ax.plot(hist.index, hist["train_loss"], marker="o", label="train"); ax.plot(hist.index, hist["val_loss"], marker="o", label="val")
ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig("q8_loss_curves.png", dpi=120); plt.show()
wandb.log({"loss_curves": wandb.Image("q8_loss_curves.png")})
wandb.finish()